***Preliminary operations***

**Import libraries**

In [ ]:
import os
import pandas as pd
from collections import Counter
import re
import nltk
from sklearn.naive_bayes import GaussianNB
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import SGDClassifier
from nltk.tokenize import word_tokenize
from collections import Counter
from imblearn.over_sampling import SMOTE
from sklearn.datasets import make_classification
from sklearn.metrics import classification_report
from numpy import where
import matplotlib.pyplot as plt 
from imblearn.over_sampling import ADASYN
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer
from wordcloud import WordCloud

**Change of directory**

In [ ]:
#change of directory

os.chdir(r'C:\Users\lenovo\Desktop\DS\llm-detect-ai-generated-text')

**Dataset upload and display**

In [ ]:
#upload of the dataset

df = pd.read_csv(r'train_essays.csv')

In [ ]:
#dataframe display

df

**EDA**

In [ ]:
#some tries with the next

df['text'][1374].index('advantages') #retrieving the index of a word
df['text'][715:735] #retrieving the text given indexes

In [ ]:
#function that counts the words' occurrances

def count_word_occurrence(text):
    
    #removes undesidered characters
    clean_text = re.sub(r'[^\w\s]', '', text)

    #divide the text in words
    words = clean_text.split()
    
    #uses a counter to count the number of words
    words_count = Counter(words)

    return words_count

In [ ]:
df

In [ ]:
copy = df.copy()

In [ ]:
#count of words occurrence

for j in df['text']:
    count = count_word_occurrence(j)
    print(count)

In [ ]:
#function that counts the occurrence of words without stopwords

def count_occurrence_words_without_stopwords(text):
    
    #removes undesired characters from text
    clean_text = re.sub(r'[^\w\s]', '', text)

    #divides the text in words
    words = clean_text.split()

    #removes
    stopwords_english = set(stopwords.words('english'))
    words_without_stopwords = [word for word in words if word.lower() not in stopwords_english]

    #uses Counter to count the occurrences of each word
    words_count = Counter(words_without_stopwords)

    return words_count, words_without_stopwords


#application of the function: iterate over dataset to count the words' occurrence wthout stopwords

list_words = []
list_words_no_stop = []

for j in range(len(df)):

    count_without_stopwords, words_no_stopwords = count_occurrence_words_without_stopwords(df['text'][j])
    list_words.append(count_without_stopwords)
    list_words_no_stop.append(words_no_stopwords)
    

#integration of list_words_no_stop into the dataset

df['words_without_stopwords'] = list_words_no_stop

In [ ]:
list_words[0]

In [ ]:
#function summing two dictionaries and their keys' occurrences

def dictionary_sum(dict1, dict2):
    
    #uses Counter to get the sum of values for duplicated keys
    total_count = Counter(dict1) + Counter(dict2)

    #converts the result in a standard dictionary
    result = dict(total_count)

    return result

#usage of the function to count to sum over dictionaries (i.e. 'text' elements) creating a list of words and their corresponding number of times that appear in the dataframe
for j in range(len(list_words)):

    if j == 0:
        result = list_words[j]
    else:
        result = dictionary_sum(result, list_words[j])

#printing of the result
print(result)


In [ ]:
#order keys on the base of values
ordered_keys = sorted(result, key=lambda k: result[k])
words_count = []
most_frequent_words = []


#printing of ordered keys
for key in ordered_keys:
    most_frequent_words.append(key) #appending of the key
    most_frequent_words.append(result[key]) #appending of the result of the key

In [ ]:
#sorting the dictionary 
sorted_dict = dict(sorted(result.items(), key=lambda item: item[1], reverse=True))

top_15_items = list(sorted_dict.items())[:15]

# Estrai chiavi e valori separatamente dalle prime 15 coppie
keys = [item[0] for item in top_15_items]
values = [item[1] for item in top_15_items]

#reshape of the axis' labels
for i, v in enumerate(values):
    plt.text(i, v + 0.5, str(v), color='black', ha='center', va='bottom', fontsize=12)


#size of the plot
plt.figure(figsize=(13, 6))

# Crea un grafico a barre
plt.barh(keys, values, color='red')


# Aggiungi etichette al grafico
plt.xlabel('Keys', fontsize=15)
plt.ylabel('Values', fontsize=15)
plt.title('First 15 words')

plt.show()

**Modeling**

In [ ]:
#top 15 keys
top_15_keys = ordered_keys[-15:]

In [ ]:
#top 15 keys     
for i in range(len(df)):
    for word in df['words_without_stopwords'][i]:
        if word in top_15_keys:
            df.at[i, word] = 1

In [ ]:
df

In [ ]:
df_freq = df.copy()

In [ ]:
df_freq

**Model Estimation**

In [ ]:
#copy of the dataset to save the results

df_copy = df.copy()
df_copy_2 = df.copy()

In [ ]:
'''There are still NaN values: they must be replaced with zeros.'''

The dataframes used for the modeling part have to be cleaned from the unuseful columns.

In [ ]:
#df with 15 stopwords
df_15 = df_copy.drop(columns=['id','text', 'words_without_stopwords'])

#df 
temp = df[['id','text', 'words_without_stopwords']]
df = df.drop(columns=['id','text', 'words_without_stopwords'])
df_copy = df_copy.drop(columns=['id','text', 'words_without_stopwords'])


In [ ]:
#filling NaNs values with zeros
df_15 = df_15.fillna(0)

In [ ]:
#NaN check
nan_check_15 = df_15.isna().sum()
nan_check_15

In [ ]:
#splitting into X and y of the dataset to estimate the model

#dataframe with 15 stopwords
y_15 = df_15['generated']
df_15 = df_15.drop(columns=['generated'])
X_15 = df_15


In [ ]:
#training and test split 

X_train, X_test, y_train, y_test = train_test_split(X_15, y_15, test_size=0.33, random_state=42)

#modeling

'''Logistic Regression'''

clf = LogisticRegression(random_state=0).fit(X_train, y_train) #classifier instantiation
y_pred = clf.predict(X_test) #prediction of the y
print(clf.score(X_test, y_test)) #scoring

score_1 = clf.score(X_test, y_test)

'''Random Forest'''

clf_2 = RandomForestClassifier(max_depth=2, random_state=0)
clf_2.fit(X_train, y_train)
y_pred_2 = clf_2.predict(X_test)
print(clf_2.score(X_test, y_test))

score_2 = clf_2.score(X_test, y_test)


'''SGD Classifier'''

clf_3 = SGDClassifier(max_iter=1000, tol=1e-3)
clf_3.fit(X_train, y_train)
y_pred_3 = clf_3.predict(X_test)
print(clf_3.score(X_test, y_test))

score_3 = clf_3.score(X_test, y_test)

'''Naive-Bayes classifier'''

clf_4 = GaussianNB()
clf_4.fit(X_train, y_train)
y_pred_4 = clf_3.predict(X_test)
print(clf_4.score(X_test, y_test))

score_4 = clf_4.score(X_test, y_test)

In [ ]:
#scores' plotting

scores = [score_1, score_2, score_3, score_4]
models = ['Random Forest', 'LogisticRegression', 'SGDClassifier', 'Naive-Bayes Classifier']

#plot dimension
plt.figure(figsize=(15, 6))

#barplot creation
plt.barh(models, scores, color='red')  #changing the 

# Aggiunta di etichette al grafico
plt.xlabel('Accuracy', fontsize=15)
plt.ylabel('Models', fontsize=15)
plt.title('Accuracy of models')

plt.show()


In [ ]:
'''The models' performance can vary with the number of words considered as (dummy) variables chosen at the beginning of the process.
This becomes particularly evident in the following sections.'''

Comparison among different solutions of number of dummy words variables considered

**Evalutation Metrics**

In [ ]:
#classification reports

print(classification_report(y_test, y_pred))
print('-------------------------------------------------------')
print(classification_report(y_test, y_pred_2))
print('-------------------------------------------------------')
print(classification_report(y_test, y_pred_3))
print('-------------------------------------------------------')
print(classification_report(y_test, y_pred_4))


**Solving the unbalanced learning problem**

In [ ]:
y = y_15
X = X_15

In [ ]:
X

In [ ]:
'''The purpose of the ADASYN algorithm is to improve class balance by synthetically creating new examples from the minority class via linear interpolation between existing minority class examples. This approach by itself is known as the SMOTE method (Synthetic Minority Oversampling TEchnique). ADASYN is an extension of SMOTE, creating more examples in the vicinity of the boundary between the two classes than in the interior of the minority class.
Since the scope of this oversampling example is to create examples that are labeled as 0 and 1, this is a more precise way to do so. '''

In [ ]:
#oversampling
oversample = ADASYN(n_neighbors=2, random_state=42)  
X_res, y_res = oversample.fit_resample(X, y)

#changing values of X_res and y_res
for column in X_res.columns:
    if column != 'prompt_id':
        for j in range(len(X_res[column])):
            if X_res[column][j] < 0.5: #assigning the observations to classes through the choice of a boundary
                X_res.at[j, column] = 0
            else: 
                X_res.at[j, column] = 1
                

In [ ]:
#model estimation

X_train, X_test, y_train, y_test = train_test_split(X_res, y_res, test_size=0.33, random_state=42)

'''Logistic Regression'''

from sklearn.linear_model import LogisticRegression
clf = LogisticRegression(random_state=0).fit(X_train, y_train)
y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)
print('Logistic Regression test score:')
print(clf.score(X_test, y_test))
score_1 = clf.score(X_test, y_test)
print('--------------------------------')

'''Random Forest'''

from sklearn.ensemble import RandomForestClassifier
clf_2 = RandomForestClassifier(max_depth=2, random_state=0)
clf_2.fit(X_train, y_train)
y_pred_2 = clf_2.predict(X_test)
y_prob_2 = clf_2.predict_proba(X_test)
print('Random Forest test score:')
print(clf_2.score(X_test, y_test))
score_2 = clf_2.score(X_test, y_test)
print('--------------------------------')

'''SGD Classifier'''

from sklearn.linear_model import SGDClassifier
clf_3 = SGDClassifier(max_iter=1000, tol=1e-3, loss='log_loss')
clf_3.fit(X_train, y_train)
y_pred_3 = clf_3.predict(X_test)
y_prob_3 = clf_3.predict_proba(X_test)
#print(y_prob_2)
print('SGD Classifier test score:')
print(clf_3.score(X_test, y_test))
score_3 = clf_3.score(X_test, y_test)
print('--------------------------------')

'''Naive-Bayes classifier'''

clf_4 = GaussianNB()
clf_4.fit(X_train, y_train)
y_pred_4 = clf_4.predict(X_test)
print('Naive-Bayes Classifier test score:')
print(clf_4.score(X_test, y_test))
score_4 = clf_4.score(X_test, y_test)


In [ ]:
#scores' plotting
scores = [score_1, score_2, score_3, score_4]
models = ['Random Forest', 'LogisticRegression', 'SGDClassifier', 'Naive-Bayes']

#plot dimension
plt.figure(figsize=(15, 6))

#barplot creation
plt.barh(models, scores, color='red')  # 

# Aggiunta di etichette al grafico
plt.xlabel('Accuracy', fontsize=15)
plt.ylabel('Models', fontsize=15)
plt.title('Accuracy of models')

plt.show()

**Classification reports**

In [ ]:
print('Classification report for Logistic Regression:')
print()
print(classification_report(y_test, y_pred))
print()
print()

print('Classification report for Random Forest:')
print()
print(classification_report(y_test, y_pred_2))
print()
print()

print('Classification report for SGD Classifier:')
print()
print(classification_report(y_test, y_pred_3))

print('Classification report for Naive-Bayes Classifier:')
print()
print(classification_report(y_test, y_pred_4))

**Removing stopwords**

In [ ]:
#application of the function: iterate over the rebalanced dataset to count the words' occurrence wthout stopwords

list_words = []
list_words_no_stop = []

for j in range(len(df)):

    count_without_stopwords, words_no_stopwords = count_occurrence_words_without_stopwords(copy['text'][j])
    list_words.append(count_without_stopwords)
    list_words_no_stop.append(words_no_stopwords)
    

#integration of list_words_no_stop into the dataset

copy['words_without_stopwords'] = list_words_no_stop

In [ ]:
copy

**Tokenizing**

In [ ]:
import nltk
from nltk.tokenize import word_tokenize
import pandas as pd

nltk.download('punkt')  #download of the punkt tokenizer data

#df is your DataFrame with a 'text' column
#create or load your DataFrame here

#creating an empty list to store the tokenized words for each row
tokenized_words_list = []

#iterating over each row in the 'text' column
for text_content in copy['text']:
    
    #tokenizing the words in the current text content
    words_in_quote = word_tokenize(text_content)
    
    #appending the tokenized words to the list
    tokenized_words_list.append(words_in_quote)
    
#adding the tokenized words as a new column in the DataFrame
copy['tokenized_words'] = tokenized_words_list

In [ ]:
filtered_list = []

nltk.download("stopwords")
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

for i in range(len(df)):
    for word in copy['tokenized_words'][i]:
        if word not in stop_words:
                
                 filtered_list.append(word)

In [ ]:
filtered_list

**Stemming and Lemmatizing**

In [ ]:
'''Stemming is a text processing task in which you reduce words to their root, which is the core part of a word.'''
'''Like stemming, lemmatizing reduces words to their core meaning, but it will give you a complete English word that makes sense on its own instead of just a fragment of a word like'''

In [ ]:
#create an empty list to store the stemmed words in a list
stemmed_words_list = []
lemmatized_words_list = []

#create a Porter Stemmer and a WordNetLemmatizer instance
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

#iterate over each row in the 'text' column
for text_content in copy['text']:
    
    #tokenize the words in the current text content
    words_in_quote = word_tokenize(text_content)
    
    #stem the words and append the stemmed words to the list
    stemmed_words = [stemmer.stem(word) for word in words_in_quote] #list comprehension
    stemmed_words_list.append(stemmed_words) #stemming
    
    #lemmatizing the words in the current text content
    lemmatizing_words = [stemmer.stem(word) for word in words_in_quote] #list comprehension
    lemmatized_words_list.append(lemmatizing_words) #lemmatizing

#add the stemmed and lemmatized words as a new column in the DataFrame
copy['stemmed_words'] = stemmed_words_list
copy['lemmatized_words'] = lemmatized_words_list

**Stemming and Lemmatizing WordCloud**

In [ ]:
#flattening the list f lists into a single list
flat_stemmed_list = [word for sublist in stemmed_words_list for word in sublist]
flat_lemmatized_list = [word for sublist in lemmatized_words_list for word in sublist]


#joining of words
stemmed_string = ' '.join(flat_stemmed_list)
lemmatized_list = ' '.join(flat_lemmatized_list)

**Stemmed words**

In [ ]:
#stemming wordcloud
wordcloud = WordCloud(width=800, height=400, background_color='white').generate(stemmed_string)

#Wordcloud of stemmed words
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.show()

**Lemmatized Words**

In [ ]:
#lemmatizing wordcloud
wordcloud = WordCloud(width=800, height=400, background_color='white').generate(lemmatized_list)

#Wordcloud show using matplotlib
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.show()

**Part-of-Speech Tagging**

In [ ]:
from nltk import pos_tag

In [ ]:
copy

In [ ]:
for i in range(len(copy)):       
        pos_tags = pos_tag(copy['tokenized_words'][i])
        print(pos_tags)

In [ ]:
from nltk import ne_chunk

named_entities = ne_chunk(pos_tags)
print(named_entities)

**Name Entity Recognition (NER)**

In [ ]:
nltk.download("maxent_ne_chunker")
nltk.download("words")

tree = nltk.ne_chunk(copy['text'])
tree.draw()

**Frequency distribution for words**

In [ ]:
from nltk import FreqDist

**No stopwords frequency distribution**

In [ ]:
copy

In [ ]:
for i in copy['words_without_stopwords']:
    frequency_distribution_no_stop = FreqDist(i)
    print(frequency_distribution_no_stop)

**Tokenized words distribution**

In [ ]:
for i in copy['tokenized_words']:
    frequency_distribution_tokenized = FreqDist(i)
    print(frequency_distribution_tokenized)

**Stemmed words distribution**

In [ ]:
for i in copy['stemmed_words']:
    frequency_distribution_stemmed = FreqDist(i)
    print(frequency_distribution_stemmed)

**Most common tokenized and stemmed words**

In [ ]:
frequency_distribution_tokenized.most_common(20)
frequency_distribution_stemmed.most_common(20)

In [ ]:
#stemmed words istogram

plt.figure(figsize=(20, 10))  #graphic dimension
plt.title('Most common stemmed words') #title
frequency_distribution_stemmed.plot(30, cumulative=False) #plot
plt.show()

In [ ]:
#tokenized words instogram

plt.figure(figsize=(20, 10))  #graphic dimension
plt.title('Most common tokenized words') #title
frequency_distribution_tokenized.plot(30, cumulative=False) #plot
plt.show()